# تحليل متوسط العمر المتوقع

مجموعتا بيانات تستكشفان متوسط العمر المتوقع حول العالم:
- **Gapminder** (1952-2007): country, year, population, continent, lifeExp, gdpPercap
- **متوسط العمر المتوقع من منظمة الصحة العالمية (WHO)** (2000-2015): 193 دولة، 22 مؤشرًا (الوفيات، مؤشر كتلة الجسم BMI، الناتج المحلي، سنوات التعليم، إلخ.)

يوضح هذا المصنف استيراد وتحليل ملفات CSV في كل من **Python** و**R**.

## 1. الإعداد: تثبيت الحزم وتنزيل مجموعات البيانات

In [ ]:
import micropip
await micropip.install(['pandas', 'plotly'])
print('تم تثبيت pandas + plotly')

import pyodide.http, os

datasets = {
    "gapminder.csv": "https://raw.githubusercontent.com/resbaz/r-novice-gapminder-files/master/data/gapminder-FiveYearData.csv",
    "who_life_expectancy.csv": "https://raw.githubusercontent.com/Sid-149/Life-Expectancy-Predictor-Comparative-Analysis/main/Notebooks/Life%20Expectancy%20Data.csv"
}

os.makedirs("/shared/data", exist_ok=True)

for name, url in datasets.items():
    path = f"/shared/data/{name}"
    if os.path.exists(path):
        print(f"موجود بالفعل: {path}")
    else:
        resp = await pyodide.http.pyfetch(url)
        text = await resp.string()
        with open(path, "w") as f:
            f.write(text)
        lines = text.count("\n")
        print(f"تم تنزيل {name}: {lines} أسطر")

## 2. Gapminder: الاستكشاف باستخدام Python

In [ ]:
import pandas as pd

gap = pd.read_csv("/shared/data/gapminder.csv")
print(f"الأبعاد: {gap.shape}")
print(f"القارات: {sorted(gap['continent'].unique())}")
print(f"نطاق السنوات: {gap['year'].min()}-{gap['year'].max()}")
print()
gap.describe()

In [ ]:
import plotly.express as px
import json, js
from plotly.utils import PlotlyJSONEncoder

def plain_plotly(value):
    if hasattr(value, 'tolist'):
        return value.tolist()
    if isinstance(value, dict):
        return {key: plain_plotly(item) for key, item in value.items()}
    if isinstance(value, (list, tuple)):
        return [plain_plotly(item) for item in value]
    return value

def show_plotly(fig):
    payload = plain_plotly(fig.to_plotly_json())
    js.renderPlot(json.dumps({"traces": payload["data"], "layout": payload["layout"]}, cls=PlotlyJSONEncoder))

# متوسط العمر المتوقع بمرور الوقت حسب القارة
avg = gap.groupby(['year', 'continent'])['lifeExp'].mean().reset_index()
fig = px.line(avg, x='year', y='lifeExp', color='continent',
              title='متوسط العمر المتوقع حسب القارة (1952-2007)',
              labels={'lifeExp': 'متوسط العمر المتوقع (بالسنوات)', 'year': 'السنة'})
fig.update_layout(template='plotly_dark')
show_plotly(fig)

In [ ]:
# الناتج المحلي الإجمالي مقابل متوسط العمر المتوقع (2007)، حجم الفقاعة = عدد السكان
g2007 = gap[gap['year'] == 2007]
fig = px.scatter(g2007, x='gdpPercap', y='lifeExp', size='pop',
                 color='continent', hover_name='country',
                 log_x=True, size_max=50,
                 title='الناتج المحلي الإجمالي مقابل متوسط العمر المتوقع (2007)',
                 labels={'gdpPercap': 'نصيب الفرد من الناتج المحلي الإجمالي (لوغاريتمي)', 'lifeExp': 'متوسط العمر المتوقع'})
fig.update_layout(template='plotly_dark')
show_plotly(fig)

## 3. Gapminder: الاستكشاف باستخدام R

In [ ]:
gap <- read.csv("/shared/data/gapminder.csv")
str(gap)
summary(gap$lifeExp)

In [ ]:
# توزيع متوسط العمر المتوقع حسب القارة (مخطط الصندوق)
par(bg = "#1e1e1e", fg = "white", col.axis = "white",
    col.lab = "white", col.main = "white")
boxplot(lifeExp ~ continent, data = gap,
        main = "متوسط العمر المتوقع حسب القارة",
        xlab = "القارة", ylab = "متوسط العمر المتوقع (بالسنوات)",
        col = c("#636EFA", "#EF553B", "#00CC96", "#AB63FA", "#FFA15A"),
        border = "white")

In [ ]:
# أفضل 10 دول من حيث تحسن متوسط العمر المتوقع (1952 مقابل 2007)
early <- gap[gap$year == 1952, c("country", "lifeExp")]
late  <- gap[gap$year == 2007, c("country", "lifeExp")]
merged <- merge(early, late, by = "country", suffixes = c("_1952", "_2007"))
merged$improvement <- merged$lifeExp_2007 - merged$lifeExp_1952
top10 <- head(merged[order(-merged$improvement), ], 10)

par(bg = "#1e1e1e", fg = "white", col.axis = "white",
    col.lab = "white", col.main = "white", mar = c(5, 10, 4, 2))
barplot(top10$improvement, names.arg = top10$country,
        horiz = TRUE, las = 1,
        main = "أفضل 10: الزيادة في متوسط العمر المتوقع (1952-2007)",
        xlab = "السنوات المكتسبة",
        col = "#00CC96", border = NA)

## 4. متوسط العمر من WHO: الاستكشاف باستخدام Python

In [ ]:
who = pd.read_csv("/shared/data/who_life_expectancy.csv")
print(f"الأبعاد: {who.shape}")
print(f"الأعمدة: {list(who.columns)}")
print(f"\nالقيم المفقودة (أعلى 5):")
print(who.isnull().sum().sort_values(ascending=False).head())
print()
who.head()

In [ ]:
# الدول النامية مقابل المتقدمة: توزيعات مجمعة مسبقًا لمتوسط العمر المتوقع
# يتم عرض إحداثيات الأشرطة الصريحة بشكل متسق عبر جسر Plotly في المتصفح.
import numpy as np
life = who.dropna(subset=['Life expectancy'])
edges = np.linspace(life['Life expectancy'].min(), life['Life expectancy'].max(), 41)
bin_width = edges[1] - edges[0]
hist_rows = []
for status, group in life.groupby('Status'):
    counts, _ = np.histogram(group['Life expectancy'], bins=edges)
    hist_rows.extend({
        'Life expectancy': float(left + bin_width / 2),
        'Count': int(count),
        'Status': status
    } for left, count in zip(edges[:-1], counts))

hist = pd.DataFrame(hist_rows)
fig = px.bar(hist, x='Life expectancy', y='Count', color='Status',
             barmode='overlay', opacity=0.7,
             title='متوسط العمر المتوقع: النامية مقابل المتقدمة',
             labels={'Life expectancy': 'متوسط العمر المتوقع (بالسنوات)'})
fig.update_traces(width=float(bin_width * 0.92))
fig.update_layout(template='plotly_dark', bargap=0.03)
show_plotly(fig)

In [ ]:
# سنوات التعليم مقابل متوسط العمر المتوقع
w2014 = who[who['Year'] == 2014].dropna(subset=['Schooling', 'Life expectancy'])
fig = px.scatter(w2014, x='Schooling', y='Life expectancy',
                 color='Status', hover_name='Country',
                 title='سنوات التعليم مقابل متوسط العمر المتوقع (2014)',
                 labels={'Life expectancy': 'متوسط العمر المتوقع (بالسنوات)',
                         'Schooling': 'سنوات التعليم'})
fig.update_layout(template='plotly_dark')
show_plotly(fig)

## 5. متوسط العمر من WHO: الاستكشاف باستخدام R

In [ ]:
who <- read.csv("/shared/data/who_life_expectancy.csv")
str(who)
cat("\nالدول:", length(unique(who$Country)))
cat("\nنطاق السنوات:", range(who$Year))

In [ ]:
# الارتباط: معدل وفيات البالغين مقابل متوسط العمر المتوقع
par(bg = "#1e1e1e", fg = "white", col.axis = "white",
    col.lab = "white", col.main = "white")
plot(who$Adult.Mortality, who$Life.expectancy,
     pch = 16, cex = 0.5,
     col = ifelse(who$Status == "Developed", "#636EFA80", "#EF553B80"),
     main = "معدل وفيات البالغين مقابل متوسط العمر المتوقع",
     xlab = "معدل وفيات البالغين (لكل 1000)",
     ylab = "متوسط العمر المتوقع (بالسنوات)")
legend("topright", legend = c("متقدمة", "نامية"),
       col = c("#636EFA", "#EF553B"), pch = 16, text.col = "white")

In [ ]:
# نموذج خطي بسيط: ما الذي يتنبأ بمتوسط العمر المتوقع؟
who_clean <- na.omit(who[, c("Life.expectancy", "Schooling",
                              "Adult.Mortality", "GDP", "BMI")])
model <- lm(Life.expectancy ~ Schooling + Adult.Mortality + log1p(GDP) + BMI,
            data = who_clean)
summary(model)

## النتائج الرئيسية

- ارتفع متوسط العمر المتوقع عالميًا ولكن لا تزال هناك فجوات كبيرة بين القارات
- يعد الناتج المحلي الإجمالي وسنوات التعليم من المؤشرات الإيجابية القوية لمتوسط العمر المتوقع
- يعد معدل وفيات البالغين أقوى مؤشر سلبي
- تُظهر الدول النامية تباينًا أوسع بكثير في النتائج